# From Notebook to Production Service

This notebook walks through the full `decider` workflow end-to-end:

1. Define modules interactively using the `%%module` magic
2. Compose and test the pipeline in the notebook
3. Save a versioned JSON config
4. Start an HTTP server with `decider serve`

In [ ]:
%pip install -q "decider[serve-sanic,notebook]"


In [1]:
import warnings; warnings.filterwarnings('ignore')
import os, json, asyncio
from pathlib import Path
import polars as pl
from pydantic import BaseModel

# All generated files land inside this notebook's folder, not wherever
# the kernel happens to be cwd'd — consistent regardless of how Jupyter is launched.
HERE = Path(__file__).parent if '__file__' in dir() else Path('.')

## Load the `%%module` magic

The `%%module` magic writes an extension file under `decider_extensions/` and
registers the class in the current session — no manual file management needed.

In [2]:
# Point the magic at this notebook's own extensions folder so generated
# files stay inside docs/examples/projects/05_notebook_to_serving/ and
# are covered by its .gitignore.
os.environ['DECIDER_EXTENSIONS_DIR'] = str(HERE / 'decider_extensions')
%load_ext decider.magics

## Define the feature engineering module

Each top-level function in the cell body becomes an output column.
Parameter names map to input column names; a `config: MyConfig` parameter
injects versioned config values.

In [3]:
%%module LoanFeatures

def debt_to_income(monthly_debt: pl.Expr, income: pl.Expr) -> pl.Expr:
    return (monthly_debt * 12) / income

def monthly_surplus(income: pl.Expr, monthly_debt: pl.Expr) -> pl.Expr:
    return income / 12 - monthly_debt

[module] Written:  /Users/cp371651/Documents/Workspace/Upskilling/dsp-decision-engine/docs/examples/projects/decider_extensions/loan_features/__init__.py
[module] LoanFeatures registered  type='loan_features'
[module] LoanFeatures injected into namespace


## Define the scoring module with config injection

`RiskConfig` fields are promoted directly onto the module — you can pass
`threshold=0.3` when instantiating, or load it from a saved JSON config.

In [4]:
%%module RiskScorer

class RiskScorerConfig(BaseModel):
    threshold: float = 0.4

def risk_score(debt_to_income: pl.Expr, monthly_surplus: pl.Expr) -> pl.Expr:
    return (debt_to_income * 0.7 + (1 / (monthly_surplus + 1)) * 0.3).round(4)

def risk_tier(risk_score: pl.Expr, config: RiskScorerConfig) -> pl.Expr:
    return pl.when(risk_score > config.threshold).then(pl.lit("HIGH")).otherwise(pl.lit("LOW"))

[module] Written:  /Users/cp371651/Documents/Workspace/Upskilling/dsp-decision-engine/docs/examples/projects/decider_extensions/risk_scorer/__init__.py
[module] RiskScorer registered  type='risk_scorer'
[module] RiskScorer injected into namespace


## Test the pipeline in the notebook

Modules registered by `%%module` are available immediately in the namespace.
Compose with `|` and call with a plain dict of Polars DataFrames.

In [5]:
applicants = pl.DataFrame({
    "customer_id":  ["C001", "C002", "C003", "C004"],
    "income":       [50_000, 35_000, 80_000, 22_000],
    "monthly_debt": [ 1_200,    900,  2_100,    700],
    "loan_amount":  [10_000, 15_000, 25_000,  5_000],
})

pipeline = LoanFeatures(name="features") | RiskScorer(name="scorer", threshold=0.4)
result = pipeline({"input": applicants})
print(result.select(["customer_id", "debt_to_income", "risk_score", "risk_tier"]))

shape: (4, 4)
┌─────────────┬────────────────┬────────────┬───────────┐
│ customer_id ┆ debt_to_income ┆ risk_score ┆ risk_tier │
│ ---         ┆ ---            ┆ ---        ┆ ---       │
│ str         ┆ f64            ┆ f64        ┆ str       │
╞═════════════╪════════════════╪════════════╪═══════════╡
│ C001        ┆ 0.288          ┆ 0.2017     ┆ LOW       │
│ C002        ┆ 0.308571       ┆ 0.2161     ┆ LOW       │
│ C003        ┆ 0.315          ┆ 0.2206     ┆ LOW       │
│ C004        ┆ 0.381818       ┆ 0.2675     ┆ LOW       │
└─────────────┴────────────────┴────────────┴───────────┘


## Save a versioned config

`asave` stages the module in memory; `save_version` writes a timestamped
JSON file to disk. The server reads this file at startup.

In [7]:
from decider.config.file import JsonFileConfigManager

CONFIGS_DIR = HERE / 'model' / 'configs'
CONFIGS_DIR.mkdir(parents=True, exist_ok=True)

mgr = JsonFileConfigManager(basepath=str(CONFIGS_DIR))
versioned = await pipeline.asave('main', mgr)
await mgr.save_version(overwrite=True)

print(f'Saved version: {versioned.version}')
print(f'Config directory: {CONFIGS_DIR}')

Saved version: 0.0.0
Config directory: /Users/cp371651/Documents/Workspace/Upskilling/dsp-decision-engine/docs/examples/projects/model/configs


## Inspect the saved config

The config is plain JSON — diff-friendly, human-readable, and
safe to check into version control.

In [8]:
version_str = str(versioned.version)
config_path = CONFIGS_DIR / version_str / 'main.json'

with open(config_path) as f:
    print(json.dumps(json.load(f), indent=2))

{
  "type": "sequential",
  "name": "features",
  "steps": [
    {
      "type": "loan_features",
      "name": "features"
    },
    {
      "threshold": 0.4,
      "type": "risk_scorer",
      "name": "scorer"
    }
  ]
}


## Start the inference server

`decider serve` picks up the config directory from the
`DECIDER_CONFIG__BASEPATH` environment variable.

The server exposes two endpoints:
- `POST /predict` — accepts JSON or Arrow IPC, returns scored output
- `GET  /ping`    — health check

> **Note:** the cell below starts a blocking process. Run it in a terminal or
> uncomment the `subprocess` variant to run it in the background.

In [ ]:
# Set the config path so the server knows where to load models from.
os.environ['DECIDER_CONFIG__BASEPATH'] = str(CONFIGS_DIR)
os.environ['DECIDER_CONFIG__TYPE']     = 'file:json'

# Run in a terminal (from this notebook's directory):
#   decider serve --engine sanic --port 8080
#
# Or start non-blocking from Python (uncomment to use):
# import subprocess, sys
# server = subprocess.Popen(
#     ['decider', 'serve', '--engine', 'sanic', '--port', '8080'],
#     env=os.environ.copy(),
# )
# import time; time.sleep(2)  # give sanic a moment to start

print('To start the server, run in a terminal:')
print(f'  DECIDER_CONFIG__BASEPATH={CONFIGS_DIR} decider serve --engine sanic --port 8080')

To start the server, run in a terminal:
  DECIDER_CONFIG__BASEPATH=/Users/cp371651/Documents/Workspace/Upskilling/dsp-decision-engine/docs/examples/projects/model/configs decider serve --engine sanic --port 8080


/opt/miniconda/envs/spockappdev/bin/python: No module named decider.cli.__main__; 'decider.cli' is a package and cannot be directly executed


## Send a prediction request

Once the server is running, call `/predict` with a JSON payload.
The server returns a JSON object with the scored columns appended.

In [ ]:
import urllib.request

payload = json.dumps({
    "customer_id":  ["C001", "C002"],
    "income":       [50_000, 35_000],
    "monthly_debt": [ 1_200,    900],
    "loan_amount":  [10_000, 15_000],
}).encode()

# Uncomment once the server is running:
# req = urllib.request.Request(
#     "http://localhost:8080/predict",
#     data=payload,
#     headers={"Content-Type": "application/json"},
# )
# with urllib.request.urlopen(req) as resp:
#     print(json.dumps(json.loads(resp.read()), indent=2))

print("Prediction payload:")
print(json.dumps(json.loads(payload), indent=2))

## Deploying with Docker

Build and run the pre-built Docker image (requires `docker/Dockerfile` and a
built wheel in `dist/`):

```bash
# Build wheel
python -m build --wheel -o dist/

# Build image
docker build -t decider:latest -f docker/Dockerfile .

# Run — mount your model directory
docker run -p 8080:8080 \
  -v $(pwd)/model:/opt/ml/model:ro \
  -e DECIDER_SERVE__WORKERS=5 \
  decider:latest
```

The server starts automatically using `decider serve --engine sanic`.
Workers default to `nproc * 2 + 1` when `DECIDER_SERVE__WORKERS` is not set.